# Temporal FitzHugh–Nagumo synchronization with binary coupling and emergency rescue

This notebook simulates a temporal directed network of identical FitzHugh–Nagumo systems and monitors their deviation from a prescribed characteristic trajectory
\[
\dot s=f(s),\qquad s(0)=(0,0)^\top .
\]

A user-selected number of vertices begin exactly on the synchronous trajectory. After \(t=0\), every vertex evolves according to the same network dynamics.

For each snapshot, a random directed graph is generated subject to weak connectivity of its underlying undirected graph. Two fixed network-wide coupling strengths,
\[
0<w_{\mathrm L}<w_{\mathrm H},
\]
are available and are selected snapshot by snapshot.

The synchronization quantity monitored is
\[
V(t)=\frac1n\sum_{i=1}^{n}\frac12\|x_i(t)-s(t)\|_2^2.
\]

The user chooses two error levels,
\[
0<\varepsilon_{\mathrm L}<\varepsilon_{\mathrm H}.
\]

The upper level \(\varepsilon_{\mathrm H}\) determines when emergency rescue is triggered. If rescue is enabled and \(V(t)\) reaches this level, a graph- and state-dependent greedy procedure selects vertices for feedback pinning. The selected vertices remain pinned until \(V(t)\le\varepsilon_{\mathrm L}\), after which all rescue inputs are removed.

The controls allow the user to choose:

- number of vertices;
- number of temporal snapshots;
- number of initially synchronized vertices;
- initial-condition spread;
- lower error level \(\varepsilon_{\mathrm L}\);
- upper error level \(\varepsilon_{\mathrm H}\);
- emergency rescue on/off;
- random seed.

The characteristic trajectory passes through a locally expansive region of the FitzHugh–Nagumo dynamics, so the aggregate error need not decay monotonically.

In [ ]:

import math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from scipy.integrate import solve_ivp

import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

np.set_printoptions(precision=5, suppress=True)


## Google Drive output

Simulation data and figures are stored directly in

`MyDrive/Temporal_FHN_Results/`

inside a run-specific folder. Vector graphics are saved in EPS format and a JPEG copy is also created for quick viewing.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_OUTPUT_DIR = Path(
        "/content/drive/MyDrive/Temporal_FHN_Results"
    )
except Exception:
    BASE_OUTPUT_DIR = Path.cwd() / "Temporal_FHN_Results"

BASE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Output directory:", BASE_OUTPUT_DIR)


## Dynamical model

Each network vertex has state \(x_i=(v_i,r_i)^\top\) and intrinsic dynamics
\[
\dot v_i=v_i-\frac{v_i^3}{3}-r_i+I,
\qquad
\dot r_i=\frac{v_i+a-br_i}{\tau}.
\]

The interaction is full-state diffusive coupling,
\[
-w\sum_{j\in N_i^-}(x_i-x_j).
\]

When emergency pinning is active at vertex \(i\), the additional feedback is
\[
-c_{\mathrm P}(x_i-s).
\]

The characteristic trajectory \(s(t)\) is integrated independently; it is not a special network vertex.

Because the coupling is diffusive, it vanishes when all network vertices have the same state. Therefore, without emergency pinning, the network is not forced to track this particular characteristic trajectory \(s(t)\). The plotted quantity \(V(t)\) can decrease, increase, or exhibit transient growth depending on the intrinsic dynamics, the random snapshot graph, the initial conditions, and the selected coupling level.


In [ ]:
FHN_A = 0.7
FHN_B = 0.8
FHN_TAU = 12.5
FHN_I = 0.5

SYNC_INITIAL_STATE = np.array([0.0, 0.0])

# Duration of one graph snapshot
DELTA_T = 0.35

# Fixed binary coupling strengths
W_LOW = 0.08
W_HIGH = 1.40

# Default error levels shown in the UI
DEFAULT_EPSILON_L = 0.020
DEFAULT_EPSILON_H = 0.100

# Emergency-pinning design
PINNING_TARGET_RATE = 0.80
DEFAULT_PIN_GAIN = 5.0

def fhn(x):
    v, r = x
    return np.array([
        v - v**3 / 3.0 - r + FHN_I,
        (v + FHN_A - FHN_B * r) / FHN_TAU,
    ])

S0 = np.array([
    [1.0, (-1.0 + 1.0 / FHN_TAU) / 2.0],
    [(-1.0 + 1.0 / FHN_TAU) / 2.0, -FHN_B / FHN_TAU],
])
RHO = float(np.linalg.eigvalsh(S0).max())

print(f"rho = {RHO:.6f}")
print(f"w_L = {W_LOW:.3f}")
print(f"w_H = {W_HIGH:.3f}")
print(f"default epsilon_L = {DEFAULT_EPSILON_L:.3f}")
print(f"default epsilon_H = {DEFAULT_EPSILON_H:.3f}")


## Random weakly connected temporal digraphs

Each snapshot is generated independently. A random spanning tree is first constructed and randomly oriented, after which additional directed arcs are inserted. This guarantees weak connectivity without imposing a leader, universal vertex, DAG structure, or common hierarchy.


In [ ]:

def generate_weakly_connected_digraph(n_vertices, rng):
    edges = set()

    order = list(rng.permutation(n_vertices))

    # Randomly oriented spanning tree
    for idx in range(1, n_vertices):
        u = int(order[idx])
        v = int(order[rng.integers(0, idx)])

        if rng.random() < 0.5:
            edges.add((u, v))
        else:
            edges.add((v, u))

    # Additional sparse directed arcs
    extra_probability = min(
        0.18,
        3.0 / max(n_vertices - 1, 1),
    )

    for u in range(n_vertices):
        for v in range(n_vertices):
            if u == v or (u, v) in edges:
                continue

            if rng.random() < extra_probability:
                edges.add((u, v))

    return sorted(edges)

def generate_snapshot_sequence(n_vertices, n_snapshots, rng):
    return [
        generate_weakly_connected_digraph(n_vertices, rng)
        for _ in range(n_snapshots)
    ]

def is_weakly_connected(n_vertices, edges):
    G = nx.DiGraph()
    G.add_nodes_from(range(n_vertices))
    G.add_edges_from(edges)
    return nx.is_weakly_connected(G)



## Initial conditions and aggregate error

The selected initially synchronized vertices satisfy
\[
x_i(0)=s(0).
\]

All remaining vertices are placed at random directions from \(s(0)\). The user-selected **initial-condition spread** determines how far they may start from the characteristic trajectory.

The spread control extends to \(10\), so experiments can begin far outside the small-error regime.

The aggregate error is
\[
V(t)
=
\frac1n
\sum_{i=1}^n
\frac12\|x_i(t)-s(t)\|_2^2.
\]


In [ ]:

def create_initial_conditions(
    n_vertices,
    initially_synchronized,
    spread,
    rng,
):
    Y0 = np.zeros((n_vertices, 2))
    Y0[:] = SYNC_INITIAL_STATE

    synchronized_vertices = tuple(
        sorted(
            rng.choice(
                n_vertices,
                size=initially_synchronized,
                replace=False,
            ).tolist()
        )
    )

    unsynchronized_vertices = [
        node
        for node in range(n_vertices)
        if node not in synchronized_vertices
    ]

    if unsynchronized_vertices:
        directions = rng.normal(
            size=(len(unsynchronized_vertices), 2)
        )

        norms = np.linalg.norm(
            directions,
            axis=1,
            keepdims=True,
        )
        norms[norms == 0.0] = 1.0
        directions /= norms

        # Use radii all the way from a moderate fraction of the selected
        # spread up to the full selected spread.
        radii = spread * rng.uniform(
            0.20,
            1.00,
            size=(len(unsynchronized_vertices), 1),
        )

        Y0[unsynchronized_vertices] = (
            SYNC_INITIAL_STATE + radii * directions
        )

    return Y0, synchronized_vertices

def local_energies(Y, s):
    difference = Y - s
    return 0.5 * np.sum(difference * difference, axis=1)

def aggregate_error(Y, s):
    return float(np.mean(local_energies(Y, s)))



## LOW/HIGH snapshot choice

For the uniform analysis vector \(p=\frac1n\mathbf 1\), a conservative scalar comparison rate is obtained from
\[
\eta_k(w)
=
\max_j
\left[
2\rho
+w\left(d_j^{\rm out}-d_j^{\rm in}\right)
\right].
\]

At the beginning of a snapshot, LOW is attempted first. If its one-snapshot exponential upper estimate stays below \(\varepsilon_{\mathrm H}\), LOW is used. Otherwise HIGH is used.

The bound is conservative: an interval labelled LOW is allowed to exhibit actual temporary growth, provided the upper tolerance is not crossed.


In [ ]:

def graph_degrees(n_vertices, edges):
    indegree = np.zeros(n_vertices, dtype=int)
    outdegree = np.zeros(n_vertices, dtype=int)

    for tail, head in edges:
        outdegree[tail] += 1
        indegree[head] += 1

    return indegree, outdegree

def comparison_column_scores(
    n_vertices,
    edges,
    coupling_gain,
):
    indegree, outdegree = graph_degrees(
        n_vertices,
        edges,
    )

    return (
        2.0 * RHO
        + coupling_gain * (outdegree - indegree)
    ).astype(float)

def scalar_rate_bound(
    n_vertices,
    edges,
    coupling_gain,
):
    return float(
        np.max(
            comparison_column_scores(
                n_vertices,
                edges,
                coupling_gain,
            )
        )
    )

def choose_binary_strength(
    Y,
    s,
    edges,
    remaining_time,
    epsilon_high,
):
    V_now = aggregate_error(Y, s)

    eta_low = scalar_rate_bound(
        Y.shape[0],
        edges,
        W_LOW,
    )

    low_estimate = (
        V_now
        * np.exp(eta_low * remaining_time)
    )

    if low_estimate <= epsilon_high:
        return "LOW", W_LOW

    return "HIGH", W_HIGH



## Emergency rescue

If rescue is enabled and \(V(t)\) reaches \(\varepsilon_{\mathrm H}\), the network switches to HIGH coupling and vertices are added greedily to the pinning set.

For the current state, vertices are ranked by their contribution to the comparison upper bound. Vertices are added until the predicted derivative satisfies
\[
\dot V\le-\chi_{\mathrm P}V.
\]

The selected rescue set stays pinned until
\[
V(t)\le\varepsilon_{\mathrm L}.
\]

If the graph changes before recovery is complete, the selection rule can add further vertices, but it does not remove already pinned vertices during the same rescue episode.


In [ ]:

def augment_pinning_set(
    Y,
    s,
    edges,
    current_pin_set,
    current_pin_gain,
):
    n_vertices = Y.shape[0]
    V_now = aggregate_error(Y, s)
    W = local_energies(Y, s)

    pin_set = set(current_pin_set)
    pin_gain = max(
        float(current_pin_gain),
        DEFAULT_PIN_GAIN,
    )

    target_bound = (
        -PINNING_TARGET_RATE * V_now
    )

    scores = comparison_column_scores(
        n_vertices,
        edges,
        W_HIGH,
    )

    def current_bound():
        modified = scores.copy()

        if pin_set:
            modified[list(pin_set)] -= (
                2.0 * pin_gain
            )

        return float(
            np.mean(modified * W)
        )

    candidates = [
        node
        for node in range(n_vertices)
        if node not in pin_set
        and W[node] > 1e-14
    ]

    while (
        current_bound() > target_bound
        and candidates
    ):
        selected = max(
            candidates,
            key=lambda node: (
                scores[node] * W[node]
            ),
        )

        pin_set.add(selected)
        candidates.remove(selected)

    if (
        current_bound() > target_bound
        and pin_set
    ):
        base_bound = float(
            np.mean(scores * W)
        )

        pinned_energy = float(
            np.sum(
                [
                    W[node]
                    for node in pin_set
                ]
            )
        )

        if pinned_energy > 1e-14:
            required_gain = (
                (base_bound - target_bound)
                * n_vertices
                / (2.0 * pinned_energy)
            )

            pin_gain = max(
                pin_gain,
                1.05 * required_gain,
            )

    return (
        tuple(sorted(pin_set)),
        pin_gain,
    )



## Temporal simulation

The ODE solver monitors \(\varepsilon_{\mathrm H}\) and \(\varepsilon_{\mathrm L}\) as events. Emergency pinning can therefore start and stop inside a snapshot rather than only at snapshot boundaries.


In [ ]:

def simulate_experiment(
    n_vertices=12,
    n_snapshots=24,
    initially_synchronized=3,
    initial_spread=0.30,
    epsilon_low=DEFAULT_EPSILON_L,
    epsilon_high=DEFAULT_EPSILON_H,
    use_emergency_pinning=True,
    random_seed=7,
):
    if not 2 <= n_vertices <= 100:
        raise ValueError(
            "n_vertices must lie between 2 and 100."
        )

    if not 2 <= n_snapshots <= 100:
        raise ValueError(
            "n_snapshots must lie between 2 and 100."
        )

    if not (
        1 <= initially_synchronized <= n_vertices
    ):
        raise ValueError(
            "initially_synchronized must lie between 1 and n_vertices."
        )
    if epsilon_low <= 0 or epsilon_high <= 0:
        raise ValueError(
            "Both error levels must be positive."
        )

    if epsilon_high <= epsilon_low:
        raise ValueError(
            "The upper error level must be greater than the lower error level."
        )

    rng = np.random.default_rng(
        random_seed
    )

    snapshots = generate_snapshot_sequence(
        n_vertices,
        n_snapshots,
        rng,
    )

    assert all(
        is_weakly_connected(
            n_vertices,
            edges,
        )
        for edges in snapshots
    )

    Y0, synchronized_vertices = (
        create_initial_conditions(
            n_vertices,
            initially_synchronized,
            initial_spread,
            rng,
        )
    )

    y_current = np.concatenate([
        Y0.ravel(),
        SYNC_INITIAL_STATE.copy(),
    ])

    V_initial = aggregate_error(
        Y0,
        SYNC_INITIAL_STATE,
    )

    time_history = [0.0]
    error_history = [V_initial]

    snapshot_labels = []
    snapshot_pin_sets = []

    pinning_transitions = []
    rescue_active = False
    active_pin_set = tuple()
    active_pin_gain = DEFAULT_PIN_GAIN

    current_time = 0.0

    for k, edges in enumerate(snapshots):
        snapshot_end = (
            (k + 1) * DELTA_T
        )

        modes_used = []
        pins_used_in_snapshot = set()

        while (
            current_time
            < snapshot_end - 1e-11
        ):
            Y = y_current[
                : 2 * n_vertices
            ].reshape(
                n_vertices,
                2,
            )

            s = y_current[
                2 * n_vertices :
            ]

            remaining_time = (
                snapshot_end
                - current_time
            )

            # If rescue is enabled and the interval begins already above
            # the high level, activate rescue immediately.
            if (
                use_emergency_pinning
                and not rescue_active
                and aggregate_error(Y, s)
                >= epsilon_high
            ):
                rescue_active = True

                pinning_transitions.append(
                    (current_time, "ON")
                )

            if rescue_active:
                (
                    active_pin_set,
                    active_pin_gain,
                ) = augment_pinning_set(
                    Y,
                    s,
                    edges,
                    active_pin_set,
                    active_pin_gain,
                )

                coupling_label = (
                    "HIGH + PINNING"
                )
                coupling_gain = W_HIGH
                pin_set_for_segment = (
                    active_pin_set
                )

            else:
                (
                    coupling_label,
                    coupling_gain,
                ) = choose_binary_strength(
                    Y,
                    s,
                    edges,
                    remaining_time,
                    epsilon_high,
                )

                pin_set_for_segment = tuple()

            modes_used.append(
                coupling_label
            )
            pins_used_in_snapshot.update(
                pin_set_for_segment
            )

            incoming = [
                []
                for _ in range(
                    n_vertices
                )
            ]

            for tail, head in edges:
                incoming[head].append(
                    tail
                )

            pin_set_for_rhs = set(
                pin_set_for_segment
            )

            def rhs(t, y):
                YY = y[
                    : 2 * n_vertices
                ].reshape(
                    n_vertices,
                    2,
                )

                ss = y[
                    2 * n_vertices :
                ]

                dY = np.zeros_like(YY)

                for node_i in range(
                    n_vertices
                ):
                    xi = YY[node_i]

                    coupling_sum = (
                        np.zeros(2)
                    )

                    for node_j in incoming[
                        node_i
                    ]:
                        coupling_sum += (
                            xi
                            - YY[node_j]
                        )

                    pinning_term = (
                        np.zeros(2)
                    )

                    if (
                        node_i
                        in pin_set_for_rhs
                    ):
                        pinning_term = (
                            active_pin_gain
                            * (xi - ss)
                        )

                    dY[node_i] = (
                        fhn(xi)
                        - coupling_gain
                        * coupling_sum
                        - pinning_term
                    )

                ds = fhn(ss)

                return np.concatenate([
                    dY.ravel(),
                    ds,
                ])

            event_functions = None

            if (
                use_emergency_pinning
                and not rescue_active
            ):
                def upper_event(t, y):
                    YY = y[
                        : 2 * n_vertices
                    ].reshape(
                        n_vertices,
                        2,
                    )

                    ss = y[
                        2 * n_vertices :
                    ]

                    return (
                        aggregate_error(
                            YY,
                            ss,
                        )
                        - epsilon_high
                    )

                upper_event.terminal = True
                upper_event.direction = 1

                event_functions = [
                    upper_event
                ]

            elif rescue_active:
                def lower_event(t, y):
                    YY = y[
                        : 2 * n_vertices
                    ].reshape(
                        n_vertices,
                        2,
                    )

                    ss = y[
                        2 * n_vertices :
                    ]

                    return (
                        aggregate_error(
                            YY,
                            ss,
                        )
                        - epsilon_low
                    )

                lower_event.terminal = True
                lower_event.direction = -1

                event_functions = [
                    lower_event
                ]

            solution = solve_ivp(
                rhs,
                (
                    current_time,
                    snapshot_end,
                ),
                y_current,
                events=event_functions,
                max_step=DELTA_T / 35.0,
                rtol=1e-8,
                atol=1e-10,
            )

            for idx in range(
                1,
                len(solution.t),
            ):
                YY = solution.y[
                    : 2 * n_vertices,
                    idx,
                ].reshape(
                    n_vertices,
                    2,
                )

                ss = solution.y[
                    2 * n_vertices :,
                    idx,
                ]

                time_history.append(
                    float(
                        solution.t[idx]
                    )
                )

                error_history.append(
                    aggregate_error(
                        YY,
                        ss,
                    )
                )

            y_current = (
                solution.y[:, -1]
            )

            current_time = float(
                solution.t[-1]
            )

            event_occurred = (
                event_functions is not None
                and len(
                    solution.t_events[0]
                ) > 0
            )

            if event_occurred:
                if rescue_active:
                    rescue_active = False
                    active_pin_set = tuple()
                    active_pin_gain = (
                        DEFAULT_PIN_GAIN
                    )

                    pinning_transitions.append(
                        (
                            current_time,
                            "OFF",
                        )
                    )

                else:
                    rescue_active = True

                    pinning_transitions.append(
                        (
                            current_time,
                            "ON",
                        )
                    )

                current_time += 1e-10

            else:
                break

        compact_modes = []

        for mode in modes_used:
            if (
                not compact_modes
                or mode
                != compact_modes[-1]
            ):
                compact_modes.append(mode)

        snapshot_labels.append(
            " -> ".join(
                compact_modes
            )
        )

        snapshot_pin_sets.append(
            tuple(
                sorted(
                    pins_used_in_snapshot
                )
            )
        )

    return {
        "time": np.asarray(
            time_history
        ),
        "V": np.asarray(
            error_history
        ),
        "snapshots": snapshots,
        "snapshot_labels": (
            snapshot_labels
        ),
        "snapshot_pin_sets": (
            snapshot_pin_sets
        ),
        "pinning_transitions": (
            pinning_transitions
        ),
        "initially_synchronized": (
            synchronized_vertices
        ),
        "initial_V": float(
            V_initial
        ),
        "final_V": float(
            error_history[-1]
        ),
        "n_snapshots": n_snapshots,
        "epsilon_low": float(epsilon_low),
        "epsilon_high": float(epsilon_high),
    }


# Interactive experiment

Use the controls below to choose the network size, number of snapshots, number of initially synchronized vertices, initial-condition spread, lower and upper error levels, emergency-rescue option, and random seed.

The error levels must satisfy
\[
0<\varepsilon_{\mathrm L}<\varepsilon_{\mathrm H}.
\]

The upper level activates emergency pinning when it is reached. The lower level determines when the rescue input is released.

In [ ]:
vertex_slider = widgets.IntSlider(
    value=12,
    min=2,
    max=100,
    step=1,
    description="Vertices:",
    continuous_update=False,
    style={"description_width": "185px"},
    layout=widgets.Layout(width="580px"),
)

snapshot_slider = widgets.IntSlider(
    value=24,
    min=2,
    max=100,
    step=1,
    description="Snapshots:",
    continuous_update=False,
    style={"description_width": "185px"},
    layout=widgets.Layout(width="580px"),
)

initial_sync_slider = widgets.IntSlider(
    value=3,
    min=1,
    max=12,
    step=1,
    description="Initially synchronized:",
    continuous_update=False,
    style={"description_width": "185px"},
    layout=widgets.Layout(width="580px"),
)

spread_slider = widgets.FloatSlider(
    value=0.30,
    min=0.05,
    max=10.00,
    step=0.05,
    description="Initial-condition spread:",
    readout_format=".2f",
    continuous_update=False,
    style={"description_width": "185px"},
    layout=widgets.Layout(width="580px"),
)

epsilon_low_widget = widgets.BoundedFloatText(
    value=DEFAULT_EPSILON_L,
    min=0.0001,
    max=100.0,
    step=0.005,
    description="Lower level epsilon_L:",
    style={"description_width": "185px"},
    layout=widgets.Layout(width="380px"),
)

epsilon_high_widget = widgets.BoundedFloatText(
    value=DEFAULT_EPSILON_H,
    min=0.0002,
    max=1000.0,
    step=0.01,
    description="Upper level epsilon_H:",
    style={"description_width": "185px"},
    layout=widgets.Layout(width="380px"),
)

rescue_checkbox = widgets.Checkbox(
    value=True,
    description="Use emergency pinning",
    indent=False,
)

seed_widget = widgets.IntText(
    value=7,
    description="Random seed:",
    style={"description_width": "185px"},
    layout=widgets.Layout(width="340px"),
)

run_button = widgets.Button(
    description="Run simulation",
    tooltip="Generate a temporal network and run the experiment",
)

output_area = widgets.Output()

LAST_RESULT = None
LAST_RUN_DIR = None
LAST_N_VERTICES = None

def update_initial_sync_limit(change=None):
    initial_sync_slider.max = vertex_slider.value

    if initial_sync_slider.value > vertex_slider.value:
        initial_sync_slider.value = vertex_slider.value

vertex_slider.observe(
    update_initial_sync_limit,
    names="value",
)

update_initial_sync_limit()

def _number_tag(value, digits=3):
    text = f"{float(value):.{digits}f}"
    return text.replace(".", "p")

def run_from_ui(_=None):
    global LAST_RESULT
    global LAST_RUN_DIR
    global LAST_N_VERTICES

    with output_area:
        clear_output(wait=True)

        n_vertices = vertex_slider.value
        n_snapshots = snapshot_slider.value
        initial_sync = initial_sync_slider.value
        spread = spread_slider.value
        epsilon_low = float(epsilon_low_widget.value)
        epsilon_high = float(epsilon_high_widget.value)
        use_rescue = rescue_checkbox.value
        random_seed = seed_widget.value

        if epsilon_low <= 0 or epsilon_high <= epsilon_low:
            print(
                "Please choose error levels satisfying "
                "0 < epsilon_L < epsilon_H."
            )
            return

        print("Running temporal FitzHugh–Nagumo experiment...")
        print(f"vertices                 : {n_vertices}")
        print(f"snapshots                : {n_snapshots}")
        print(f"initially synchronized   : {initial_sync}")
        print(f"initial-condition spread : {spread:.2f}")
        print(f"lower level epsilon_L    : {epsilon_low:.4f}")
        print(f"upper level epsilon_H    : {epsilon_high:.4f}")
        print(f"fixed low coupling w_L   : {W_LOW:.4f}")
        print(f"fixed high coupling w_H  : {W_HIGH:.4f}")
        print(f"emergency pinning        : {use_rescue}")
        print(f"random seed              : {random_seed}")

        result = simulate_experiment(
            n_vertices=n_vertices,
            n_snapshots=n_snapshots,
            initially_synchronized=initial_sync,
            initial_spread=spread,
            epsilon_low=epsilon_low,
            epsilon_high=epsilon_high,
            use_emergency_pinning=use_rescue,
            random_seed=random_seed,
        )

        folder_name = (
            f"run_n{n_vertices}"
            f"_snap{n_snapshots}"
            f"_sync{initial_sync}"
            f"_spread{_number_tag(spread)}"
            f"_epsL{_number_tag(epsilon_low)}"
            f"_epsH{_number_tag(epsilon_high)}"
            f"_seed{random_seed}"
        )

        run_dir = BASE_OUTPUT_DIR / folder_name
        run_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        np.savez_compressed(
            run_dir / "simulation_data.npz",
            time=result["time"],
            V=result["V"],
        )

        with open(
            run_dir / "run_summary.txt",
            "w",
            encoding="utf-8",
        ) as summary_file:
            summary_file.write(
                f"vertices: {n_vertices}\n"
                f"snapshots: {n_snapshots}\n"
                f"initially synchronized: {initial_sync}\n"
                f"initially synchronized vertices: "
                f"{result['initially_synchronized']}\n"
                f"initial-condition spread: {spread}\n"
                f"w_L: {W_LOW}\n"
                f"w_H: {W_HIGH}\n"
                f"epsilon_L: {epsilon_low}\n"
                f"epsilon_H: {epsilon_high}\n"
                f"emergency pinning: {use_rescue}\n"
                f"random seed: {random_seed}\n"
                f"initial V: {result['initial_V']}\n"
                f"final V: {result['final_V']}\n"
                f"snapshot actions:\n"
                + " | ".join(result["snapshot_labels"])
                + "\n"
            )

        LAST_RESULT = result
        LAST_RUN_DIR = run_dir
        LAST_N_VERTICES = n_vertices

        print(
            "initially synchronized vertices:",
            result["initially_synchronized"],
        )
        print(f"initial V = {result['initial_V']:.6e}")
        print(f"final V   = {result['final_V']:.6e}")

        print("snapshot actions:")
        print(" | ".join(result["snapshot_labels"]))

        print("\nSimulation data saved to:")
        print(run_dir)
        print(
            "\nUse the figure cells below to generate "
            "the aggregate-error figure and snapshot images."
        )

run_button.on_click(run_from_ui)

controls = widgets.VBox([
    vertex_slider,
    snapshot_slider,
    initial_sync_slider,
    spread_slider,
    widgets.HBox([
        epsilon_low_widget,
        epsilon_high_widget,
    ]),
    rescue_checkbox,
    seed_widget,
    run_button,
])

display(
    controls,
    output_area,
)

## Aggregate synchronization-error figure

Run this cell after the simulation. The plot displays the complete aggregate error \(V(t)\), the lower level \(\varepsilon_L\), the upper level \(\varepsilon_H\), snapshot coupling labels, and pinning transitions.

The legend is placed outside the plotting area. The figure is saved directly to the current Google Drive run folder in EPS and JPEG formats.

In [ ]:
def generate_aggregate_error_figure():
    if LAST_RESULT is None or LAST_RUN_DIR is None:
        print("Run the simulation first.")
        return

    result = LAST_RESULT
    t = result["time"]
    V = result["V"]
    n_snapshots = result["n_snapshots"]
    epsilon_low = result["epsilon_low"]
    epsilon_high = result["epsilon_high"]

    fig, ax = plt.subplots(
        figsize=(15, 7.5)
    )

    ax.plot(
        t,
        V,
        linewidth=2.2,
        label="Aggregate error V(t)",
    )

    ax.axhline(
        epsilon_high,
        linestyle=(0, (7, 3, 7, 3, 7, 3)),
        linewidth=1.6,
        label=f"Upper level epsilon_H = {epsilon_high:.3f}",
    )

    ax.axhline(
        epsilon_low,
        linestyle="-.",
        linewidth=1.6,
        label=f"Lower level epsilon_L = {epsilon_low:.3f}",
    )

    boundary_times = (
        np.arange(n_snapshots + 1)
        * DELTA_T
    )

    for boundary in boundary_times[1:-1]:
        ax.axvline(
            boundary,
            linewidth=0.45,
            linestyle=":",
            alpha=0.45,
        )

    label_stride = max(
        1,
        int(np.ceil(n_snapshots / 28)),
    )

    for k, label in enumerate(result["snapshot_labels"]):
        if (
            k % label_stride != 0
            and "PINNING" not in label
        ):
            continue

        midpoint = (
            (k + 0.5)
            * DELTA_T
        )

        shortened = label.replace(
            "HIGH + PINNING",
            "HIGH+PIN",
        )

        ax.text(
            midpoint,
            0.965,
            shortened,
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=7.8,
            fontweight="bold",
            rotation=90 if len(shortened) > 8 else 0,
            clip_on=True,
        )

    for transition_time, state in result["pinning_transitions"]:
        ax.axvline(
            transition_time,
            linewidth=1.4,
            linestyle="-.",
        )

        ax.text(
            transition_time,
            0.76,
            "PINNING " + state,
            transform=ax.get_xaxis_transform(),
            rotation=90,
            ha="right",
            va="top",
            fontsize=8.5,
        )

    ax.set_xlabel("Time")
    ax.set_ylabel("V(t)")
    ax.set_title(
        "Aggregate synchronization error",
        pad=24,
    )
    ax.grid(alpha=0.20)

    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
    )

    fig.subplots_adjust(
        right=0.76
    )

    eps_path = LAST_RUN_DIR / "aggregate_synchronization_error.eps"
    jpeg_path = LAST_RUN_DIR / "aggregate_synchronization_error.jpeg"

    fig.savefig(
        eps_path,
        format="eps",
        bbox_inches="tight",
    )

    fig.savefig(
        jpeg_path,
        format="jpeg",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print("Saved:")
    print(eps_path)
    print(jpeg_path)

generate_aggregate_error_figure()

## Snapshot digraph grid

Run this cell after the simulation. All temporal snapshots are arranged into a single grid figure and saved directly in the current Google Drive run folder.

The first snapshot identifies the vertices that begin on the synchronous trajectory. During rescue intervals, the vertices receiving pinning feedback in each snapshot are drawn with square markers.

The exported files are:

- `all_snapshot_digraphs_grid.eps`
- `all_snapshot_digraphs_grid.jpeg`

The figure uses one panel per snapshot and places all panels into a single image.

In [ ]:
def _snapshot_positions(n_vertices):
    angles = np.linspace(
        0.0,
        2.0 * np.pi,
        n_vertices,
        endpoint=False,
    )

    return {
        node: np.array([
            np.cos(angles[node]),
            np.sin(angles[node]),
        ])
        for node in range(n_vertices)
    }

def draw_single_snapshot(
    result,
    n_vertices,
    snapshot_index,
    save=False,
):
    k = snapshot_index
    snapshots = result["snapshots"]
    labels = result["snapshot_labels"]
    pin_sets = result["snapshot_pin_sets"]

    if not 0 <= k < len(snapshots):
        raise IndexError("Invalid snapshot index.")

    initially_synchronized = set(
        result["initially_synchronized"]
    )

    pinned = set(pin_sets[k])

    G = nx.DiGraph()
    G.add_nodes_from(range(n_vertices))
    G.add_edges_from(snapshots[k])

    pos = _snapshot_positions(n_vertices)

    ordinary = [
        node
        for node in range(n_vertices)
        if node not in pinned
    ]

    fig, ax = plt.subplots(figsize=(7.2, 7.2))

    nx.draw_networkx_edges(
        G,
        pos,
        ax=ax,
        arrows=True,
        arrowsize=12 if n_vertices <= 30 else 7,
        width=0.9 if n_vertices <= 30 else 0.42,
        connectionstyle="arc3,rad=0.035",
    )

    if ordinary:
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=ordinary,
            node_shape="o",
            node_size=220 if n_vertices <= 30 else 45,
            ax=ax,
        )

    if pinned:
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=sorted(pinned),
            node_shape="s",
            node_size=320 if n_vertices <= 30 else 80,
            linewidths=1.7,
            ax=ax,
        )

    if k == 0 and initially_synchronized:
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=sorted(initially_synchronized),
            node_shape="*",
            node_size=410 if n_vertices <= 30 else 110,
            linewidths=1.2,
            ax=ax,
        )

    if n_vertices <= 25:
        nx.draw_networkx_labels(
            G,
            pos,
            labels={node: str(node) for node in range(n_vertices)},
            font_size=8,
            ax=ax,
        )

    pin_text = (
        ", ".join(str(node) for node in sorted(pinned))
        if pinned else "none"
    )

    ax.set_title(
        f"Snapshot {k + 1}: {labels[k]}\n"
        f"Pinned vertices: {pin_text}",
        pad=16,
    )
    ax.set_axis_off()
    fig.tight_layout()

    if save and LAST_RUN_DIR is not None:
        eps_path = LAST_RUN_DIR / f"snapshot_{k + 1:03d}.eps"
        jpeg_path = LAST_RUN_DIR / f"snapshot_{k + 1:03d}.jpeg"
        fig.savefig(eps_path, format="eps", bbox_inches="tight")
        fig.savefig(jpeg_path, format="jpeg", dpi=300, bbox_inches="tight")

    plt.show()

def generate_snapshot_grid_figure():
    if LAST_RESULT is None or LAST_RUN_DIR is None:
        print("Run the simulation first.")
        return

    result = LAST_RESULT
    n_vertices = LAST_N_VERTICES
    snapshots = result["snapshots"]
    labels = result["snapshot_labels"]
    pin_sets = result["snapshot_pin_sets"]
    initially_synchronized = set(result["initially_synchronized"])

    n_snapshots = len(snapshots)
    ncols = int(np.ceil(np.sqrt(n_snapshots)))
    nrows = int(np.ceil(n_snapshots / ncols))

    panel_width = 4.0
    panel_height = 3.8

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(panel_width * ncols, panel_height * nrows),
    )

    axes = np.atleast_1d(axes).ravel()
    pos = _snapshot_positions(n_vertices)

    for ax_idx, ax in enumerate(axes):
        if ax_idx >= n_snapshots:
            ax.set_axis_off()
            continue

        k = ax_idx
        G = nx.DiGraph()
        G.add_nodes_from(range(n_vertices))
        G.add_edges_from(snapshots[k])

        pinned = set(pin_sets[k])
        ordinary = [
            node
            for node in range(n_vertices)
            if node not in pinned
        ]

        nx.draw_networkx_edges(
            G,
            pos,
            ax=ax,
            arrows=True,
            arrowsize=10 if n_vertices <= 30 else 6,
            width=0.8 if n_vertices <= 30 else 0.38,
            connectionstyle="arc3,rad=0.035",
        )

        if ordinary:
            nx.draw_networkx_nodes(
                G,
                pos,
                nodelist=ordinary,
                node_shape="o",
                node_size=170 if n_vertices <= 30 else 38,
                ax=ax,
            )

        if pinned:
            nx.draw_networkx_nodes(
                G,
                pos,
                nodelist=sorted(pinned),
                node_shape="s",
                node_size=260 if n_vertices <= 30 else 70,
                linewidths=1.5,
                ax=ax,
            )

        if k == 0 and initially_synchronized:
            nx.draw_networkx_nodes(
                G,
                pos,
                nodelist=sorted(initially_synchronized),
                node_shape="*",
                node_size=320 if n_vertices <= 30 else 95,
                linewidths=1.2,
                ax=ax,
            )

        if n_vertices <= 18:
            nx.draw_networkx_labels(
                G,
                pos,
                labels={node: str(node) for node in range(n_vertices)},
                font_size=7,
                ax=ax,
            )

        short_label = labels[k].replace("HIGH + PINNING", "HIGH+PIN")
        ax.set_title(
            f"Snapshot {k + 1}\n{short_label}",
            fontsize=9,
            pad=8,
        )
        ax.set_axis_off()

    fig.suptitle(
        "Temporal snapshot digraphs",
        fontsize=16,
        y=0.995,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.985])

    eps_path = LAST_RUN_DIR / "all_snapshot_digraphs_grid.eps"
    jpeg_path = LAST_RUN_DIR / "all_snapshot_digraphs_grid.jpeg"

    fig.savefig(
        eps_path,
        format="eps",
        bbox_inches="tight",
    )
    fig.savefig(
        jpeg_path,
        format="jpeg",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print("Saved:")
    print(eps_path)
    print(jpeg_path)

generate_snapshot_grid_figure()

## Snapshot viewer

This cell displays one selected snapshot at a time. It is useful for examining a particular temporal digraph in more detail after the full grid figure has been generated.

In [ ]:
if LAST_RESULT is None:
    print("Run the simulation first.")
else:
    snapshot_view_slider = widgets.IntSlider(
        value=1,
        min=1,
        max=len(LAST_RESULT["snapshots"]),
        step=1,
        description="Snapshot:",
        continuous_update=False,
        style={"description_width": "100px"},
        layout=widgets.Layout(width="500px"),
    )

    snapshot_view_button = widgets.Button(
        description="Display snapshot"
    )

    snapshot_view_output = widgets.Output()

    def show_selected_snapshot(_=None):
        with snapshot_view_output:
            clear_output(wait=True)

            draw_single_snapshot(
                LAST_RESULT,
                LAST_N_VERTICES,
                snapshot_view_slider.value - 1,
                save=False,
            )

    snapshot_view_button.on_click(show_selected_snapshot)

    display(
        widgets.VBox([
            snapshot_view_slider,
            snapshot_view_button,
        ]),
        snapshot_view_output,
    )